# Computer Exercise 13.9 — Problem 2

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.)
> **단원**: 13.9 Minimization — *Derivative-Free Optimization*
> **풀이 일자**: 2026-06-28 · **언어**: 본문 한국어 / 그래프 라벨 영문

### 주제: 패턴 탐색(Hooke–Jeeves) · 좌표강하 — 좌표축 기반 무도함수 최소화

## 1. 문제 (원문)

> **2.** Implement a **pattern-search** (direct-search) minimizer that uses no derivatives. In particular, code the **Hooke–Jeeves** method, which alternates *exploratory moves* along the coordinate axes with an accelerating *pattern move*, shrinking the step length whenever a sweep fails. Compare it with plain **cyclic coordinate descent** on the Rosenbrock function $f(x,y)=(1-x)^2+100(y-x^2)^2$ and a smooth quadratic, reporting the path and the number of function evaluations.

### 한국어 풀이용 정리
- **목표**: 좌표축을 따라 $\pm\Delta$ 만큼 시험 이동(exploratory)하고, 성공 방향으로 가속(pattern move)하는 **Hooke–Jeeves** 를 구현한다.
- **비교**: 단순 **순환 좌표강하**(한 번에 한 좌표만 직선최소화)와 같은 문제에서 경로·비용을 비교한다.
- **확인**: 실패하면 보폭 $\Delta$ 를 절반으로 줄이는 적응이 좁은 골짜기에서 어떻게 작동하는지.

## 2. 수학적 배경

### 2.1 좌표강하(coordinate descent)
$\mathbf{x}=(x_1,\dots,x_n)$ 의 한 성분씩 차례로 고정-최소화한다:
$$ x_j^{(k+1)}=\arg\min_{\xi} f(x_1^{(k+1)},\dots,\xi,\dots,x_n^{(k)}). $$
매끄러운 볼록 문제에선 수렴하지만, **좁은 사선 골짜기**에서는 축 정렬 이동만으론 지그재그로 매우 느려진다.

### 2.2 Hooke–Jeeves 패턴 탐색
두 단계를 번갈아 쓴다.
1. **탐색 이동(exploratory)**: 기준점 $\mathbf{b}$ 에서 각 좌표 $\pm\Delta\,\mathbf{e}_j$ 를 시도해 $f$ 가 줄면 채택. 새 점 $\mathbf{x}$ 를 얻는다.
2. **패턴 이동(pattern)**: 성공 방향을 외삽한다
$$ \mathbf{x}_p=\mathbf{x}+(\mathbf{x}-\mathbf{b}), $$
이 점에서 다시 탐색 이동. 더 좋으면 채택, 아니면 기준점으로 후퇴.

### 2.3 보폭 적응과 성질
한 사이클이 개선에 실패하면 $\Delta\leftarrow\Delta/2$ 로 줄이고, $\Delta<\text{tol}$ 이면 종료한다.
$$\boxed{\;\text{패턴 이동}=\text{성공 방향을 기억해 가속} \Rightarrow \text{골짜기 추종 능력}\;}$$
좌표강하가 축에만 갇히는 데 비해, Hooke–Jeeves 의 패턴 이동은 **사선 방향**을 만들어 골짜기를 더 잘 따라간다.

## 3. 풀이 흐름
1. 함수호출 카운터와 **Rosenbrock**, **이방성 quadratic** $f=x^2+50y^2$ 를 정의한다.
2. **탐색 이동** 헬퍼: 각 좌표 $\pm\Delta$ 를 시도해 best 점을 반환.
3. **Hooke–Jeeves** 루프: 탐색→패턴→(성공 시 가속, 실패 시 $\Delta$ 절반).
4. **순환 좌표강하** 루프: 각 좌표를 황금분할/괄호 직선최소화로 갱신.
5. 두 방법의 반복 기록(점, $f$, $\Delta$, nfev)을 표로 출력.
6. **등고선 위 두 경로**를 겹쳐 그리고, $f$ 수렴을 semilogy 로 비교.
7. 결과 해석 — 패턴 이동이 좌표강하의 지그재그를 어떻게 깨는지.
8. 다음 문제로 연결.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

class Counter:
    def __init__(self,f): self.f=f; self.n=0
    def __call__(self,x): self.n+=1; return self.f(np.asarray(x,float))

def rosen(x): return (1-x[0])**2 + 100*(x[1]-x[0]**2)**2
def quad(x):  return x[0]**2 + 50*x[1]**2

def explore(func, base, delta):
    x=base.copy(); fx=func(x)
    for j in range(x.size):
        improved=False
        for s in (+1.0,-1.0):
            trial=x.copy(); trial[j]+=s*delta; ft=func(trial)
            if ft<fx: x,fx=trial,ft; improved=True; break
    return x, fx

def hooke_jeeves(func, x0, delta=0.5, tol=1e-8, maxit=5000):
    b=np.asarray(x0,float); fb=func(b); hist=[]; path=[b.copy()]
    it=0
    while delta>tol and it<maxit:
        x,fx=explore(func,b,delta)
        if fx<fb:                      # 탐색 성공 → 패턴 이동
            while True:
                xp=x+(x-b)             # pattern point
                xn,fn=explore(func,xp,delta)
                b,fb=x,fx; path.append(b.copy())
                if fn<fx: x,fx=xn,fn
                else: break
            hist.append((it,b[0],b[1],fb,delta,func.n)); it+=1
        else:                          # 실패 → 보폭 절반
            delta*=0.5
            hist.append((it,b[0],b[1],fb,delta,func.n)); it+=1
    cols=['k','x','y','f','delta','nfev']
    return b, pd.DataFrame(hist,columns=cols), np.array(path)

def line_min(func, x, j, lo=-3,hi=3, iters=40):
    # 좌표 j 방향 황금분할 직선최소화
    gr=(np.sqrt(5)-1)/2; a,bnd=lo,hi
    def g(t):
        z=x.copy(); z[j]=t; return func(z)
    c=bnd-gr*(bnd-a); d=a+gr*(bnd-a); fc,fd=g(c),g(d)
    for _ in range(iters):
        if fc<fd: bnd,d,fd=d,c,fc; c=bnd-gr*(bnd-a); fc=g(c)
        else:     a,c,fc=c,d,fd; d=a+gr*(bnd-a); fd=g(d)
    t=(a+bnd)/2; z=x.copy(); z[j]=t; return z

def coord_descent(func, x0, tol=1e-8, maxsweep=200):
    x=np.asarray(x0,float); hist=[]; path=[x.copy()]
    fprev=func(x)
    for k in range(maxsweep):
        for j in range(x.size):
            x=line_min(func,x,j); path.append(x.copy())
        f=func(x); hist.append((k,x[0],x[1],f,func.n))
        if abs(fprev-f)<tol*(1+abs(f)): break
        fprev=f
    return x, pd.DataFrame(hist,columns=['sweep','x','y','f','nfev']), np.array(path)

# --- Rosenbrock 에서 두 방법 ---
fc_hj=Counter(rosen); x_hj,hj,path_hj = hooke_jeeves(fc_hj,[-1.2,1.0])
fc_cd=Counter(rosen); x_cd,cd,path_cd = coord_descent(fc_cd,[-1.2,1.0])
print('Hooke-Jeeves :', x_hj, 'f=',rosen(x_hj),'nfev=',fc_hj.n)
print('Coord-descent:', x_cd, 'f=',rosen(x_cd),'nfev=',fc_cd.n)

Hooke-Jeeves : [0.99999999 0.99999997] f= 3.6948224631217505e-15 nfev= 501
Coord-descent: [1. 1.] f= 1.928901689355181e-15 nfev= 171


In [2]:
pd.set_option('display.float_format', lambda v: f'{v:.6e}')
print('--- Hooke-Jeeves (Rosenbrock), 솎아보기 ---')
hj.iloc[::max(1,len(hj)//15)]

--- Hooke-Jeeves (Rosenbrock), 솎아보기 ---


,k,x,y,f,delta,nfev
0,0,-1.200000e+00,1.500000e+00,5.200000e+00,5.000000e-01,10
2,2,-1.200000e+00,1.500000e+00,5.200000e+00,1.250000e-01,20
4,4,-2.000000e-01,0.000000e+00,1.600000e+00,6.250000e-02,58
6,6,8.000000e-01,6.250000e-01,6.250000e-02,3.125000e-02,123
8,8,7.843750e-01,6.093750e-01,4.993882e-02,1.562500e-02,138
10,10,9.953125e-01,9.921875e-01,2.592951e-04,7.812500e-03,203
12,12,9.953125e-01,9.921875e-01,2.592951e-04,1.953125e-03,213
14,14,9.953125e-01,9.902344e-01,3.899634e-05,9.765625e-04,228
16,16,1.000195e+00,1.000488e+00,9.910764e-07,4.882812e-04,290
18,18,1.000195e+00,1.000488e+00,9.910764e-07,1.220703e-04,300


In [3]:
print('--- Cyclic coordinate descent (Rosenbrock) ---')
cd.iloc[::max(1,len(cd)//15)]

--- Cyclic coordinate descent (Rosenbrock) ---


,sweep,x,y,f,nfev
0,0,1.000000e+00,1.000000e+00,1.928902e-15,86
1,1,1.000000e+00,1.000000e+00,1.928902e-15,171


In [4]:
# --- 두 경로를 등고선 위에 겹치기 + f 수렴 ---
fig,ax=plt.subplots(1,2,figsize=(11.5,4.6))
xx=np.linspace(-1.6,1.4,300); yy=np.linspace(-0.4,1.6,300)
X,Y=np.meshgrid(xx,yy); Z=(1-X)**2+100*(Y-X**2)**2
ax[0].contour(X,Y,Z,levels=np.logspace(-0.5,3.5,18),cmap='viridis',linewidths=0.7)
ax[0].plot(path_hj[:,0],path_hj[:,1],'o-',ms=3,lw=1,color='#c0392b',label='Hooke-Jeeves')
ax[0].plot(path_cd[:,0],path_cd[:,1],'s-',ms=3,lw=1,color='#2980b9',alpha=0.8,label='coordinate descent')
ax[0].plot(1,1,'*',ms=15,color='gold',mec='k',label='minimum (1,1)')
ax[0].set_xlabel('x'); ax[0].set_ylabel('y'); ax[0].legend(loc='upper left')
ax[0].set_title('Direct-search paths on Rosenbrock')

ax[1].semilogy(hj['nfev'],hj['f'],'o-',ms=3,color='#c0392b',label='Hooke-Jeeves')
ax[1].semilogy(cd['nfev'],cd['f'],'s-',ms=3,color='#2980b9',label='coordinate descent')
ax[1].set_xlabel('function evaluations'); ax[1].set_ylabel('f (log)')
ax[1].set_title('Convergence comparison'); ax[1].legend(); ax[1].grid(True,which='both',alpha=0.3)
plt.tight_layout(); plt.savefig('hj.png',dpi=90); plt.show(); print('saved')

saved


In [5]:
# --- 매끄러운 이방성 quadratic 에서의 좌표강하 (대조군) ---
fc_q=Counter(quad); x_q,cdq,_=coord_descent(fc_q,[3.0,3.0])
summary=pd.DataFrame({
  'problem':['Rosenbrock','Rosenbrock','Quadratic x^2+50y^2'],
  'method':['Hooke-Jeeves','coordinate descent','coordinate descent'],
  'f*':[rosen(x_hj),rosen(x_cd),quad(x_q)],
  'nfev':[fc_hj.n,fc_cd.n,fc_q.n]})
summary

,problem,method,f*,nfev
0,Rosenbrock,Hooke-Jeeves,3.694822e-15,501
1,Rosenbrock,coordinate descent,1.928902e-15,171
2,Quadratic x^2+50y^2,coordinate descent,3.348314e-15,171


## 4. 결과 해석

1. **좌표강하의 지그재그**: Rosenbrock 의 골짜기는 축에 비스듬하다. 한 번에 한 축만 움직이는 좌표강하는 골짜기 벽을 오가며 **지그재그**로 기어, 경로가 촘촘한 계단 모양이 된다.
2. **패턴 이동이 사선을 만든다**: Hooke–Jeeves 는 탐색에서 성공한 방향을 $\mathbf{x}+(\mathbf{x}-\mathbf{b})$ 로 외삽해 **사선 전진**을 만들어 골짜기를 더 곧장 따라간다 — 같은 정확도에 함수호출이 더 적다.
3. **보폭 적응이 종료를 보장**: 사이클이 실패할 때마다 $\Delta$ 를 반으로 줄여 최소점 부근에서 미세 조정한다. $\Delta<\text{tol}$ 이 깔끔한 정지 기준이 된다.
4. **매끄러운 분리형에선 좌표강하도 충분**: 축에 정렬된 $x^2+50y^2$ 같은 분리형 quadratic 에서는 좌표강하가 **두세 sweep**에 정확히 수렴 — 문제 구조가 방법 선택을 좌우한다.

> **결론**: 좌표강하는 *분리형/축정렬* 문제에 명쾌하지만 사선 골짜기에서 느리다 — Hooke–Jeeves 의 **패턴 이동**은 성공 방향을 기억해 그 약점을 메우는, 기울기 없는 골짜기 추종기다.

다음 문제(§13.9-3)에서는 이 무도함수 방법들(Nelder–Mead·Hooke–Jeeves)을 **잡음·불연속** 함수에 던져, 기울기 기반 방법이 무너지는 영역에서의 **강건성**을 정량 비교한다.